# Global Reference Frame Alignment

This notebook reproduces the full analysis pipeline for global reference
frame alignment and validation. All core algorithms are implemented in
external modules located in `src/`.

In [1]:
import numpy as np
import pytransform3d.transformations as pt
import pytransform3d.plot_utils as pu
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from src.interpolation import upsample_trajectories
from src.frames import transform_from_points
from src.rigid_alignment import optimal_tracking
from src.io import load_dataset, print_dataset_summary
from src.synchronization import estimate_shift, shift_poses
from src.handeye import estimate_handeye_shah
from src.validation import rotation_translation_error, evaluate_alignment_metrics, sweep_tau_shah_metrics
from src.plotting import plot_transform_validation, plot_shah_boxplots, plot_tau_shah_metrics, plot_tau_shah_metrics_2x2

## Dataset loading

The dataset is stored in a single `.npz` container and includes checkerboard geometry, marker trajectories, video-based trajectories, and sampling frequencies. The printed summary below is used to verify data integrity.

In [2]:
DATASET_PATH = "data/GlobalRef_DATASET.npz"

data = load_dataset(DATASET_PATH)
print_dataset_summary(data)

objp = data["checkerboard_model"]
triangulated_frames = data["video_stereo_trajectories"]
reconstructed_points_A = data["video_monocular_trajectories"]
VIDEO_FS = data["video_fs"]

markerbased_static_trajectories = data["marker_static_recording"]
markerbased_trajectories = data["marker_trajectories"]
MARKER_FS = data["marker_fs"]

Loaded dataset summary:

Variable                       Shape                Type           
----------------------------------------------------------------------
checkerboard_model             (21, 3)              ndarray        
video_stereo_trajectories      (900, 21, 3)         ndarray        
video_monocular_trajectories   (900, 21, 3)         ndarray        
video_fs                       -                    float          
marker_static_recording        (4, 3)               ndarray        
marker_trajectories            (1500, 4, 3)         ndarray        
marker_fs                      -                    float          


## Temporal upsampling

Marker-based and video-based trajectories may have different sampling frequencies. All trajectories are resampled to a common frequency defined as the maximum between the two, using cubic spline interpolation to ensure a consistent temporal grid.

In [3]:
MAX_FS = max(VIDEO_FS, MARKER_FS)

triangulated_frames_up, _ = upsample_trajectories(
    triangulated_frames, VIDEO_FS, MAX_FS
)

reconstructed_points_A_up, _ = upsample_trajectories(
    reconstructed_points_A, VIDEO_FS, MAX_FS
)

markerbased_trajectories_up, _ = upsample_trajectories(
    markerbased_trajectories, MARKER_FS, MAX_FS
)

## Pose construction

Each frame is converted into an $SE(3)$ pose by defining a local reference frame from three non-collinear points. For marker-based data, rigid alignment to a static template is applied to reduce noise. For video-based data, the checkerboard model is used as the rigid reference.

### Marker-based pose sequence

For each frame, the dynamic marker constellation is rigidly aligned to the static template and used to construct a local $SE(3)$ reference frame. The resulting sequence is treated as the reference trajectory.

In [4]:
markerbased_poses = []

for frame in markerbased_trajectories_up:
    denoised = optimal_tracking(markerbased_static_trajectories, frame)
    pts = denoised.reshape(-1, 3)

    T = transform_from_points(pts[0], pts[1], pts[2])
    markerbased_poses.append(T)

print(f"Computed {len(markerbased_poses)} marker-based poses")

Computed 1500 marker-based poses


### Stereo pose sequence

Stereo triangulation provides 3D checkerboard points per frame. Each frame is rigidly aligned to the checkerboard model and converted into an $SE(3)$ pose using a consistent set of corners.

In [5]:
video_triangulated_poses = []

for X in triangulated_frames_up:
    denoised = optimal_tracking(objp, X)
    pts = denoised.reshape(-1, 3)

    T = transform_from_points(pts[0], pts[7], pts[1])
    video_triangulated_poses.append(T)

print(f"Computed {len(video_triangulated_poses)} stereo poses")

Computed 1499 stereo poses


### Monocular pose sequence

Monocular poses are reconstructed from pose estimation. No additional rigid denoising is applied, and the same point triplet used in the stereo case is adopted for frame definition.

In [6]:
video_monocular_poses = []

for X in reconstructed_points_A_up:
    pts = X.reshape(-1, 3)
    T = transform_from_points(pts[0], pts[7], pts[1])
    video_monocular_poses.append(T)

print(f"Computed {len(video_monocular_poses)} monocular poses")

Computed 1499 monocular poses


## Temporal synchronization

A temporal offset $\tau$ is estimated via cross-correlation of a rotation-invariant signal derived from relative rotations. Synchronization is performed by shifting one timeline and interpolating $SE(3)$ poses using ScLERP.

In [7]:
tau_stereo, _ = estimate_shift(markerbased_poses, video_triangulated_poses, fs=MAX_FS)
tau_mono, _   = estimate_shift(markerbased_poses, video_monocular_poses, fs=MAX_FS)

marker_sync, stereo_sync, fs_sync, t_sync = shift_poses(
    markerbased_poses, video_triangulated_poses, MAX_FS, tau_stereo
)

_, mono_sync, _, _ = shift_poses(
    markerbased_poses, video_monocular_poses, MAX_FS, tau_mono
)

## Hand–eye calibration

After synchronization, the rigid transformation between trajectories is estimated using robot–world and hand–eye calibration with the Shah method. The resulting transforms are used for validation.

In [8]:
X_stereo, Y_stereo = estimate_handeye_shah(
    marker_sync,
    stereo_sync
)

X_mono, Y_mono = estimate_handeye_shah(
    marker_sync,
    mono_sync
)

## Rigid error validation

Alignment accuracy is first evaluated against a known ground-truth transform using geodesic rotation error and Euclidean translation error.

In [9]:
# Known transformation (ground truth)
R_known = np.eye(3)
t_known = np.array([60, 40, -4])  # mm

err_rot_st, err_trans_st = rotation_translation_error(
    X_stereo[:3, :3], X_stereo[:3, 3],
    R_known, t_known
)

err_rot_m, err_trans_m = rotation_translation_error(
    X_mono[:3, :3], X_mono[:3, 3],
    R_known, t_known
)

print("Triangulation:")
print(f" Rotation error:     {err_rot_st:.3f} deg")
print(f" Translation error:  {err_trans_st:.3f} mm\n")

print("Monocular:")
print(f" Rotation error:     {err_rot_m:.3f} deg")
print(f" Translation error:  {err_trans_m:.3f} mm")

Triangulation:
 Rotation error:     0.169 deg
 Translation error:  1.833 mm

Monocular:
 Rotation error:     0.572 deg
 Translation error:  2.295 mm


In [10]:
plot_transform_validation(
    T_known=pt.transform_from(R_known, t_known),
    T_est_stereo=X_stereo,
    T_est_mono=X_mono,
    output_path="figures/checkerboard_validation.png",
    unit="mm"
)

## Shah alignment metrics

Alignment consistency is further quantified using Shah orientation and position accuracy metrics computed over the synchronized trajectories and summarized as distributions over time.

In [11]:
ori_st, pos_st = evaluate_alignment_metrics(
    marker_sync,
    stereo_sync,
    X_stereo,
    Y_stereo
)

ori_m, pos_m = evaluate_alignment_metrics(
    marker_sync,
    mono_sync,
    X_mono,
    Y_mono
)

plot_shah_boxplots(
    ori_data=[ori_st, ori_m],
    pos_data=[pos_st, pos_m],
    labels=["Stereo", "Monocular"],
    output_path="figures/boxplot_metrics.png"
)

c:\Users\loren\OneDrive\Documenti\Work\GlobalRef\GlobalRef_Code\GlobalRef\src\plotting.py:206: UserWarning: FixedFormatter should only be used together with FixedLocator
  axes[0].set_xticklabels(labels)
c:\Users\loren\OneDrive\Documenti\Work\GlobalRef\GlobalRef_Code\GlobalRef\src\plotting.py:227: UserWarning: FixedFormatter should only be used together with FixedLocator
  axes[1].set_xticklabels(labels)


## Sensitivity analysis with respect to $\tau$

Shah metrics are evaluated over a bounded interval of temporal offsets to assess robustness of synchronization and to verify that the selected $\tau$ lies near an optimal region.

In [12]:
tau_min = -0.0170-0.01      # absolute range lower bound [s]
tau_max = -0.0170+0.01      # absolute range upper bound [s]
N = 151                     # number of tau samples
taus = np.linspace(tau_min, tau_max, N)

ori_mean_st, ori_std_st, pos_mean_st, pos_std_st = sweep_tau_shah_metrics(
    video_triangulated_poses,
    markerbased_poses,
    MAX_FS,
    taus
)

ori_mean_m, ori_std_m, pos_mean_m, pos_std_m = sweep_tau_shah_metrics(
    video_monocular_poses,
    markerbased_poses,
    MAX_FS,
    taus
)

plot_tau_shah_metrics(
    taus,
    ori_mean_st, ori_std_st,
    pos_mean_st, pos_std_st,
    ori_mean_m, ori_std_m,
    pos_mean_m, pos_std_m,
    output_path="figures/shah_metrics_compact.png",
    tau_stereo=tau_stereo,
    tau_mono=tau_mono,
)

plot_tau_shah_metrics_2x2(
    taus,
    ori_mean_st, ori_std_st,
    pos_mean_st, pos_std_st,
    ori_mean_m, ori_std_m,
    pos_mean_m, pos_std_m,
    output_path="figures/shah_metrics_extended.png",
    tau_stereo=tau_stereo,
    tau_mono=tau_mono,
)

Tau differences
 Orientation Stereo: 0.674514 ms
 Position    Stereo: 0.274514 ms
 Orientation Monocular: 3.6900 ms
 Position    Monocular: 2.4900 ms
